# 08 : Folium

강의자료: `docs/course/08_Folium.md`

GeoJSON으로 국가 경계를 읽고 ISO-3 코드로 통계를 붙인 뒤 단계구분도를 만든다.
**조인 키가 어긋나는 문제를 진단하는 과정**이 이 강의의 핵심이다.

## 데이터

- `data/raw/world-countries.geojson` — 177개국 경계. `id`가 ISO-3 코드
- `data/raw/world-centroids.geojson` — 중심점. `representative_point()`로 직접 생성
- `px.data.gapminder()` — 내장

> 강의자료에 두 GeoJSON의 출처 URL이 없다. folium 공식 저장소 예제본을 썼고
> 구조(177행, id/name/geometry)는 강의자료와 일치한다. 강사 확인이 필요하다.

In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import folium
import branca.colormap as cm
from pathlib import Path

HTML = Path("outputs/html"); HTML.mkdir(parents=True, exist_ok=True)
FIG = Path("outputs/figures"); FIG.mkdir(parents=True, exist_ok=True)

gap07 = px.data.gapminder().query("year == 2007")
print("gapminder 2007:", gap07.shape)

## 1. GeoDataFrame 읽기

In [ ]:
world = gpd.read_file("data/raw/world-countries.geojson")
print("shape:", world.shape)
print("CRS:", world.crs)
print("\n도형 종류:\n", world.geom_type.value_counts(), sep="")
world.head(3)

## 2. 지도로 확인

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
world.plot(ax=axes[0], color="white", edgecolor="black", linewidth=0.3)
axes[0].set_title("세계 국가 경계")
axes[0].axis("off")

world[world["id"] == "KOR"].plot(ax=axes[1], color="#1f6feb", edgecolor="black")
axes[1].set_title("대한민국 (KOR)")
axes[1].axis("off")

fig.tight_layout()
fig.savefig(FIG / "6_1.png", dpi=120)
plt.show()

## 3. 조인 — 행 수가 늘어나는 문제

경계 177행에 통계를 붙였는데 결과가 178행이 된다. 키가 중복됐다는 뜻이다.

In [ ]:
merged = world.merge(gap07, left_on="id", right_on="iso_alpha", how="left")
print(f"경계 {len(world)}행 + 통계 → 조인 결과 {len(merged)}행")

if len(merged) != len(world):
    print("행이 늘었다. 오른쪽 테이블에 키가 중복된 것이다.")
    dup = gap07[gap07["iso_alpha"].duplicated(keep=False)]
    print("\n중복된 iso_alpha:")
    print(dup[["country", "iso_alpha", "lifeExp", "pop"]])

In [ ]:
# 매칭되지 않은 국가 확인
print("통계가 안 붙은 국가 수:", merged["lifeExp"].isna().sum())
print("\n예시:")
print(merged[merged["lifeExp"].isna()]["name"].head(10).tolist())

## 4. 원인 수정

북한(Korea, Dem. Rep.)의 ISO 코드가 남한과 같은 `KOR`로 기록돼 있다.
북한의 정확한 코드는 `PRK`다.

In [ ]:
gap_fixed = gap07.copy()
gap_fixed.loc[gap_fixed["country"] == "Korea, Dem. Rep.", "iso_alpha"] = "PRK"

wg = world.merge(gap_fixed, left_on="id", right_on="iso_alpha", how="left")
print(f"수정 후: {len(world)}행 → {len(wg)}행")
print("중복 남았나:", gap_fixed["iso_alpha"].duplicated().any())

print("\n한국:")
print(wg[wg["id"].isin(["KOR", "PRK"])][["id", "name", "country", "lifeExp", "gdpPercap"]])

## 5. Plotly choropleth — 같은 자료를 다른 도구로

학습목표에 있는 항목이다. Folium과 비교하면 차이가 드러난다.
Plotly는 국가 코드만 주면 경계를 알아서 그리고, Folium은 GeoJSON을 직접 준다.

In [ ]:
fig = px.choropleth(
    gap_fixed, locations="iso_alpha", color="gdpPercap",
    hover_name="country", color_continuous_scale="Blues",
    projection="natural earth",
    labels={"gdpPercap": "1인당 GDP"},
)
fig.update_layout(title="국가별 1인당 GDP (2007) — Plotly",
                  height=520, margin=dict(l=0, r=0, t=50, b=0))
fig.write_html(HTML / "8_1.html", include_plotlyjs="cdn")
fig.show()

In [ ]:
# 기대수명으로도 그려 비교한다. 색 방향이 GDP와 비슷하게 가는지 본다.
fig = px.choropleth(
    gap_fixed, locations="iso_alpha", color="lifeExp",
    hover_name="country", color_continuous_scale="RdYlGn",
    projection="natural earth",
    labels={"lifeExp": "기대수명"},
)
fig.update_layout(title="국가별 기대수명 (2007) — Plotly",
                  height=520, margin=dict(l=0, r=0, t=50, b=0))
fig.write_html(HTML / "8_2.html", include_plotlyjs="cdn")
fig.show()

**Plotly와 Folium 중 무엇을 쓸까**

| | Plotly choropleth | Folium |
| --- | --- | --- |
| 경계 데이터 | 내장 (ISO 코드만 주면 됨) | 직접 준비해야 함 |
| 배경 지도 | 없음 (투영법만 선택) | 실제 타일 지도 위에 얹음 |
| 레이어 중첩 | 제한적 | 자유롭게 쌓음 |
| 적합한 경우 | 국가 단위 빠른 확인 | 행정동처럼 자체 경계가 필요한 경우 |

수업 후반 서울 행정동 분석에는 내장 경계가 없으므로 Folium 쪽이 맞다.

## 6. Folium 단계구분도

In [ ]:
geo = json.loads(wg.to_json())

m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=geo,
    data=wg,
    columns=["id", "gdpPercap"],
    key_on="feature.properties.id",
    fill_color="Blues",
    fill_opacity=0.75,
    line_opacity=0.3,
    nan_fill_color="lightgrey",     # 통계가 없는 국가는 회색으로 구분
    legend_name="1인당 GDP (달러, 2007)",
).add_to(m)

# 투명 레이어를 위에 얹어 마우스를 올리면 값이 보이게 한다
folium.GeoJson(
    geo,
    style_function=lambda _: {"fillColor": "transparent", "color": "transparent"},
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "gdpPercap", "lifeExp"],
        aliases=["국가", "1인당 GDP", "기대수명"],
        localize=True,
    ),
).add_to(m)

m.save(HTML / "9_1.html")
m

## 7. 중심점 레이어 얹기

단계구분도 위에 원을 올려 두 변수를 한 지도에서 본다.
색은 기대수명, 크기도 기대수명으로 이중 부호화해 읽기 쉽게 했다.

In [ ]:
cent = gpd.read_file("data/raw/world-centroids.geojson")
print("중심점:", cent.shape)

# inner join — 양쪽 모두에 있는 것만 남는다. left와 달리 행이 줄어든다
cg = cent.merge(gap_fixed, left_on="id", right_on="iso_alpha", how="inner")
print(f"inner join: {len(cent)} → {len(cg)}행")

In [ ]:
colormap = cm.LinearColormap(
    colors=["#c0392b", "#f0c419", "#1a7f45"],
    vmin=float(cg["lifeExp"].min()), vmax=float(cg["lifeExp"].max()),
    caption="기대수명 (년)",
)


def life_to_radius(life):
    """기대수명을 원 반지름으로 바꾼다. 40년을 기준으로 차이를 벌린다."""
    return max(3, (life - 40) * 0.45)


m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=geo, data=wg, columns=["id", "gdpPercap"],
    key_on="feature.properties.id", fill_color="Blues",
    fill_opacity=0.55, line_opacity=0.25, nan_fill_color="lightgrey",
    legend_name="1인당 GDP (달러)",
).add_to(m)

for _, row in cg.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=life_to_radius(row["lifeExp"]),
        color=colormap(row["lifeExp"]),
        fill=True, fill_color=colormap(row["lifeExp"]), fill_opacity=0.85,
        weight=1,
        tooltip=f"{row['country']}<br>기대수명 {row['lifeExp']:.1f}년"
                f"<br>1인당 GDP {row['gdpPercap']:,.0f}달러",
    ).add_to(m)

colormap.add_to(m)
m.save(HTML / "9_2.html")
m

In [ ]:
print("저장된 파일:")
for p in sorted(HTML.glob("9_*.html")) + sorted(FIG.glob("6_*.png")):
    print(" ", p)

### 이 강의에서 남길 것

조인은 붙였다고 끝이 아니다. **행 수가 변했는지 반드시 확인한다.**

- 행이 **늘면** 오른쪽 키에 중복이 있다 (여기서는 북한·남한이 둘 다 KOR)
- 행이 **줄면** inner join이라 한쪽에만 있는 것이 빠졌다
- 행 수는 같은데 값이 비면 키 표기가 다르다

수업 후반 팀 프로젝트에서 행정동 코드로 데이터를 붙일 때 똑같은 문제가 생긴다.
행정동 코드는 개편이 잦아서 연도가 다르면 코드가 안 맞는다.